In [1]:
import openai 
import instructor 
from qdrant_client import QdrantClient
from pydantic import BaseModel, Field

In [2]:
def preprocess_review(review):
    #  0   main_category   2000 non-null   object 
    #  1   title           2026 non-null   object 
    #  2   average_rating  2026 non-null   float64
    #  3   rating_number   2026 non-null   int64  
    #  4   features        2026 non-null   object 
    #  5   description     2026 non-null   object 
    #  6   price           2026 non-null   float64
    #  7   images          2026 non-null   object 
    #  8   videos          2026 non-null   object 
    #  9   store           2026 non-null   object 
    #  10  categories      2026 non-null   object 
    #  11  details         2026 non-null   object 
    #  12  parent_asin     2026 non-null   object 
    #  13  product_id      2026 non-null   object
    return {
        'product_id': review['product_id'], 
        'main_category': review['main_category'],
        'average_rating': review['average_rating'],
        'rating_number': review['rating_number'],
        'features': review['features'],
        'description': review['description'],
        'price': review['price'],
        'images': review['images'],
        'videos': review['videos'],
        'store': review['store'],
        'categories': review['categories'],
        'details': review['details'],
        'parent_asin': review['parent_asin'],
    }

In [3]:
prompt = """
You are a helful assistant.
Return an answer to the question.
Question: What is the capital of France?
"""

In [4]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
if qdrant_url and "qdrant:6333" in qdrant_url:
    # Docker service host is not resolvable from a local notebook kernel
    qdrant_url = qdrant_url.replace("qdrant:6333", "localhost:6333")

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")
print(f"Langsmith API Key present: {bool(langsmith_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False
Langsmith API Key present: True


In [5]:
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "system", "content": prompt}],
   temperature=0.7
)

In [6]:
print(response)

ChatCompletion(id='chatcmpl-DhuFboC2bZKF7mIakLXoeiNFllYlJ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1779356703, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_e4b564fb2a', usage=CompletionUsage(completion_tokens=7, prompt_tokens=31, total_tokens=38, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [7]:
print(response.choices[0].message.content)

The capital of France is Paris.


Add Instructor for Structured Outputs

In [8]:
client = instructor.from_openai(openai.OpenAI())

In [9]:
client

In [10]:
class RagGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "system", "content": prompt}],
    response_model=RagGenerationResponse,
    temperature=0.3
)
print(response)

answer='The capital of France is Paris.'


In [11]:
response = client.chat.completions.create_with_completion(
    model="gpt-4.1-mini",
    messages=[{"role": "system", "content": prompt}],
    response_model=RagGenerationResponse,
    temperature=0.3
)
print(response)

(RagGenerationResponse(answer='The capital of France is Paris.'), ChatCompletion(id='chatcmpl-DhuFeqVosX3Cyo0dr66h2IGUQ4LIW', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_pm4ZpycbU7VRblN4zT60omEq', function=Function(arguments='{"answer":"The capital of France is Paris."}', name='RagGenerationResponse'), type='function')]))], created=1779356706, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_ccea8f1341', usage=CompletionUsage(completion_tokens=11, prompt_tokens=97, total_tokens=108, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))))


In [12]:
class RagGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")
    reasoning: str = Field(description="The reasoning behind the answer")
response = client.chat.completions.create_with_completion(
    model="gpt-4.1-mini",
    messages=[{"role": "system", "content": prompt}],
    response_model=RagGenerationResponse,
    temperature=0.3
)
print(response)

(RagGenerationResponse(answer='Paris', reasoning='Paris is the capital city of France. It is well-known as the political, cultural, and economic center of the country.'), ChatCompletion(id='chatcmpl-DhuFfjuEKqWgXtShc9efgu9jPgeWA', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_iBc0eY2cuiD1rWn61sX3gel9', function=Function(arguments='{"answer":"Paris","reasoning":"Paris is the capital city of France. It is well-known as the political, cultural, and economic center of the country."}', name='RagGenerationResponse'), type='function')]))], created=1779356707, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_68f974a107', usage=CompletionUsage(completion_tokens=34, prompt_tokens=112, total_tokens=146, completion_tokens_details=CompletionTokens

Rag Example

In [13]:
class RagGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")
    reasoning: str = Field(description="The reasoning behind the answer")

In [14]:
qdrant_client = QdrantClient(
    url="http://localhost:6333",
)

In [15]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding


def retrieve_data(query, qdrant_client=qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon_Electronics_Data_Collection",
        query=query_embedding,
        limit=k,
        with_payload=True,
    )

    print("results.points: ", results.points)

    retrieved_context_ids = []
    retrieve_context = []
    similarity_scores = []
    retrieved_context_rating_numbers = []
    retrieve_context_main_categories = []
    retrieve_context_prices = []
    retrieve_context_images = []
    retrieve_context_videos = []
    retrieve_context_stores = []
    retrieve_context_categories = []
    retrieve_context_details = []
    retrieve_context_descriptions = []
    retrieve_context_features = []

    for result in results.points:
        payload = result.payload or {}
        retrieved_context_ids.append(result.id)
        similarity_scores.append(result.score)
        retrieve_context.append(payload.get('text', ''))
        retrieved_context_rating_numbers.append(payload.get('rating_number', None))
        retrieve_context_main_categories.append(payload.get('main_category', None))
        retrieve_context_prices.append(payload.get('price', None))
        retrieve_context_images.append(payload.get('images', None))
        retrieve_context_videos.append(payload.get('videos', None))
        retrieve_context_stores.append(payload.get('store', None))
        retrieve_context_categories.append(payload.get('categories', None))
        retrieve_context_details.append(payload.get('details', None))
        retrieve_context_descriptions.append(payload.get('description', None))
        retrieve_context_features.append(payload.get('features', None))

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieve_context': retrieve_context,
        'similarity_scores': similarity_scores,
        'retrieved_context_rating_numbers': retrieved_context_rating_numbers,
        'retrieve_context_main_categories': retrieve_context_main_categories,
        'retrieve_context_prices': retrieve_context_prices,
        'retrieve_context_images': retrieve_context_images,
        'retrieve_context_videos': retrieve_context_videos,
        'retrieve_context_stores': retrieve_context_stores,
        'retrieve_context_categories': retrieve_context_categories,
        'retrieve_context_details': retrieve_context_details,
        'retrieve_context_descriptions': retrieve_context_descriptions,
        'retrieve_context_features': retrieve_context_features,
    }

def process_context(context):
    formatted_context = []
    count = len(context.get('retrieved_context_ids', []))

    for index in range(count):
        point_id = context['retrieved_context_ids'][index]
        text = context['retrieve_context'][index] if index < len(context.get('retrieve_context', [])) else ''
        score = context['similarity_scores'][index] if index < len(context.get('similarity_scores', [])) else None
        rating_number = context['retrieved_context_rating_numbers'][index] if index < len(context.get('retrieved_context_rating_numbers', [])) else None
        main_category = context['retrieve_context_main_categories'][index] if index < len(context.get('retrieve_context_main_categories', [])) else ''
        price = context['retrieve_context_prices'][index] if index < len(context.get('retrieve_context_prices', [])) else None
        images = context['retrieve_context_images'][index] if index < len(context.get('retrieve_context_images', [])) else []
        videos = context['retrieve_context_videos'][index] if index < len(context.get('retrieve_context_videos', [])) else []
        store = context['retrieve_context_stores'][index] if index < len(context.get('retrieve_context_stores', [])) else ''
        categories = context['retrieve_context_categories'][index] if index < len(context.get('retrieve_context_categories', [])) else []
        details = context['retrieve_context_details'][index] if index < len(context.get('retrieve_context_details', [])) else ''
        description = context['retrieve_context_descriptions'][index] if index < len(context.get('retrieve_context_descriptions', [])) else ''
        features = context['retrieve_context_features'][index] if index < len(context.get('retrieve_context_features', [])) else []

        formatted_context.append(
            f"ID: {point_id}\n"
            f"Score: {score}\n"
            f"Rating Number: {rating_number}\n"
            f"Main Category: {main_category}\n"
            f"Price: {price}\n"
            f"Store: {store}\n"
            f"Categories: {categories}\n"
            f"Details: {details}\n"
            f"Description: {description}\n"
            f"Features: {features}\n"
            f"Images: {images}\n"
            f"Videos: {videos}\n"
            f"Text: {text}\n---"
        )

    return "\n".join(formatted_context)


def build_prompt(preprocessed_context, question):
    prompt = f"""You are a helpful assistant for answering questions about
      Amazon product reviews. Use the following retrieved context 
      to answer the question. If the context does not contain relevant information, 
      say you don't know.
      Instructions:
       - Use the retrieved context to answer the question.
       - If the context does not contain relevant information, say you don't know.

    Context:
      {preprocessed_context}
      Question: {question}
      Answer:"""
    return prompt


def gen_answer(prompt):
    # Call may return different shapes depending on client used (OpenAI SDK or a helper
    # that returns a Pydantic model). Handle both cases and normalize to a dict.
    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_model=RagGenerationResponse
    )

    return response, raw_response
   


def rag_pipeline(question, qdrant_client, top_k=5):
    retrieve_context = retrieve_data(question, qdrant_client=qdrant_client, k=top_k)
    preprocessed_context = process_context(retrieve_context)
    prompt = build_prompt(preprocessed_context, question)
    gen, raw_gen = gen_answer(prompt)

    # Normalize final answer whether gen is dict-like or an object
    if isinstance(gen, dict):
        answer_text = gen.get("text") or gen.get("answer") or str(gen)
        rag_generation_response = gen
    else:
        answer_text = getattr(gen, "answer", getattr(gen, "text", str(gen)))
        # try to convert to dict if pydantic object provides model_dump
        try:
            rag_generation_response = gen.model_dump() if hasattr(gen, "model_dump") else gen.dict()
        except Exception:
            rag_generation_response = str(gen)

    final_result = {
        "question": question,
        "answer": answer_text.answer if isinstance(answer_text, RagGenerationResponse) else answer_text,
        "retrieved_context": retrieve_context,
        "rag_generation_response": rag_generation_response,
    }

    return final_result


In [16]:
rag_pipeline("What are the best laptops you suggest to buy?", qdrant_client=QdrantClient(url=qdrant_url, api_key=qdrant_api_key), top_k=5)

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_11407/2257610999.py:1: UserWarning: Api key is used with an insecure connection.
  rag_pipeline("What are the best laptops you suggest to buy?", qdrant_client=QdrantClient(url=qdrant_url, api_key=qdrant_api_key), top_k=5)


results.points:  [ScoredPoint(id=173, version=18, score=0.4437421, payload={'product_id': '8010052f-0ed9-427c-a98a-0467d5b8b1f1', 'text': 'CUK HP Pavilion 17 Touch Gaming Notebook (i7-7700HQ, 16GB DDR4 RAM, 1TB SSHD, NVIDIA GTX 1050 4GB, 17.3" Full HD Touchscreen, Windows 10) Best Laptop for Media Students Photographers Photoshop', 'description': [], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/51bbXcvjqlL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/51bbXcvjqlL._AC_.jpg', 'variant': 'MAIN', 'hi_res': 'https://m.media-amazon.com/images/I/61TonVMnqKL._AC_SL1200_.jpg'}, {'thumb': 'https://m.media-amazon.com/images/I/41huf2znUWL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41huf2znUWL._AC_.jpg', 'variant': 'PT01', 'hi_res': 'https://m.media-amazon.com/images/I/61kKkMY9JRL._AC_SL1200_.jpg'}, {'thumb': 'https://m.media-amazon.com/images/I/41V0IaCTEpL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41V0IaCTEpL._AC_.jpg', 'variant':

{'question': 'What are the best laptops you suggest to buy?',
 'answer': 'Based on the provided context, here are some of the best laptops suggested to buy: 1. GIGABYTE AERO 16 XE5-73US948HP: A high-end laptop with a 16" 4K/UHD+ Samsung AMOLED display, Intel Core i7-12700H processor, NVIDIA GeForce RTX 3070 Ti GPU, 32GB DDR5 RAM, and 2TB SSD storage. It runs Windows 11 Pro and is suitable for demanding tasks and creative work. 2. HP 2023 Pavilion x360 14" FHD Touchscreen 2-in-1 Laptop: Features a 12th Gen Intel Core i5-1235U processor, 8GB DDR4 RAM, 1TB PCIe SSD, and Intel Iris Xe Graphics. It has a 14" multitouch-enabled IPS display and runs Windows 11. This laptop is versatile with a 360° flip-and-fold convertible design. 3. CUK HP Pavilion 17 Touch Gaming Notebook: Comes with an Intel i7-7700HQ processor, 16GB DDR4 RAM, 1TB SSHD, NVIDIA GTX 1050 4GB dedicated graphics, and a 17.3" Full HD touchscreen. It runs Windows 10 and is good for media students, photographers, and Photoshop us